<table style="border:0; background:none; width:100%">
<tr>
<td style="vertical-align: middle; width: 60%;">
<b>Pontificia Universidad Javeriana</b><br/>
Facultad de Ingeniería · Departamento de Ingeniería de Sistemas<br/>
<b>Procesamiento de Datos a Gran Escala</b>
</td>
<td style="vertical-align: middle; text-align: right; width: 40%;">
<b>Proyecto de Investigación — Entrega 2</b><br/>
Brecha digital y resultados Saber 11 en Colombia<br/>
<b>Grupo: REST pAPIs</b>
</td>
</tr>
</table>

---

# 04 — Silver: SISBEN (DNP) — agregado a nivel municipio

El SISBEN viene a nivel persona (4.5M filas). Para joins con ICFES / Internet / MEN
necesitamos **agregar a nivel municipio**.

## FILTROS APLICADOS (a nivel persona, antes de agregar)

| # | Filtro | Justificación |
|---|---|---|
| F1 | `cod_mpio IS NOT NULL` | El código DANE de municipio es la **clave de agregación** y la **clave de join** con todas las demás tablas. Una persona sin `cod_mpio` no puede ser contabilizada en su municipio y rompería los joins downstream. |
| F2 | `Grupo IN ('A','B','C','D')` | El SISBEN clasifica oficialmente a las personas en estos 4 grupos. Cualquier valor distinto (`NULL`, `''`, `Z`, etc.) es **error de captura** y contaminaría las métricas `pct_grupo_*`. |

## TRANSFORMACIONES APLICADAS

| # | Transformación | Justificación |
|---|---|---|
| T1 | Cast `I1..I15` (15 indicadores) → IntegerType | Bronze los preserva como string; necesitamos numéricos para calcular `avg(I_k)` y el índice de privación. |
| T2 | `lpad(cod_mpio, 5, '0')` → `COD_MPIO` (string) | Códigos DANE son siempre 5 dígitos; mantener como string evita perder el cero inicial (ej. `5001` → `05001`). |
| T3 | `Grupo → UPPER` ; cast `ZONA` → IntegerType ; cast `FEX` → DoubleType | Estandarización de tipos para agregaciones consistentes. |
| T4 | **Agregación masiva** persona → municipio: `groupBy(COD_MPIO).agg(...)` con n_personas, n_hogares, fex_total, pct_rural, pct_grupo_A/B/C/D, avg_I1..I15 | Reduce 4.5M personas → ~1100 municipios, grano necesario para join con ICFES/Internet/MEN. Las métricas de grupo y ruralidad se calculan como promedio de indicador booleano (técnica estándar para % a partir de microdatos). |
| T5 | Derivación `idx_privacion = avg_I1 + ... + avg_I15` | **Proxy de pobreza multidimensional** a nivel municipio (rango 0..15). Variable derivada clave para el ML. |

**Variables clave de entrada:**
- `cod_mpio`: código DANE.
- `I1..I15`: 15 indicadores binarios de privación (1 = privado).
- `Grupo` (A=más pobre, D=menos), `Nivel`, `Clasificacion` (e.g. A1, B2).
- `ZONA` (1=urbana, 2=rural).
- `FEX`: factor de expansión poblacional.

## 1. Setup y carga

In [1]:
import sys
sys.path.insert(0, "/home/estudiante/proyecto_datos/scripts")
from common_spark import get_spark, P
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

spark = get_spark("Entrega2-Silver-SISBEN", executor_memory="4g", driver_memory="2g", cores=2)
df = spark.read.parquet(P.BRONZE_PQ_SISBEN)
print(f"Filas persona: {df.count():,}   Cols: {len(df.columns)}")
print("Cols clave :", [c for c in df.columns if c in ("cod_mpio","H_5","Grupo","Nivel","Clasificacion","ZONA","HOGAR","FEX")])
df.select("cod_mpio","Grupo","Nivel","Clasificacion","ZONA","HOGAR","FEX","I1","I2","I15").show(5)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/22 08:15:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Filas persona: 4,465,955   Cols: 48
Cols clave : ['cod_mpio', 'H_5', 'Grupo', 'Nivel', 'Clasificacion', 'ZONA', 'HOGAR', 'FEX']


+--------+-----+-----+-------------+----+-----+------------+---+---+---+
|cod_mpio|Grupo|Nivel|Clasificacion|ZONA|HOGAR|         FEX| I1| I2|I15|
+--------+-----+-----+-------------+----+-----+------------+---+---+---+
|   05001|    C|    3|           C3|   1|    1|761.19639279|  1|  0|  0|
|   05001|    C|    3|           C3|   1|    1|761.19639279|  1|  0|  0|
|   05001|    C|    8|           C8|   2|    1|23.954887218|  1|  0|  0|
|   05001|    C|    8|           C8|   2|    1|23.954887218|  1|  0|  0|
|   05001|    C|    8|           C8|   2|    1|23.954887218|  1|  0|  0|
+--------+-----+-----+-------------+----+-----+------------+---+---+---+
only showing top 5 rows



## 2. Tipificar y normalizar (T1, T2, T3)

In [2]:
indicadores = [f"I{i}" for i in range(1, 16)]
df_n = df
for c in indicadores:
    df_n = df_n.withColumn(c, F.col(c).cast(IntegerType()))
df_n = (df_n
    .withColumn("COD_MPIO", F.lpad(F.col("cod_mpio"), 5, "0"))
    .withColumn("FEX",      F.col("FEX").cast(DoubleType()))
    .withColumn("ZONA",     F.col("ZONA").cast(IntegerType()))
    .withColumn("Grupo",    F.upper(F.col("Grupo")))
)
df_n.select("COD_MPIO","Grupo","ZONA","FEX","I1","I15").show(5)

+--------+-----+----+------------+---+---+
|COD_MPIO|Grupo|ZONA|         FEX| I1|I15|
+--------+-----+----+------------+---+---+
|   05001|    C|   1|761.19639279|  1|  0|
|   05001|    C|   1|761.19639279|  1|  0|
|   05001|    C|   2|23.954887218|  1|  0|
|   05001|    C|   2|23.954887218|  1|  0|
|   05001|    C|   2|23.954887218|  1|  0|
+--------+-----+----+------------+---+---+
only showing top 5 rows



## 3. Aplicación de filtros (F1 + F2)

In [3]:
n0 = df_n.count()
print(f"Personas antes de filtros: {n0:,}")

# F1 — cod_mpio no nulo (es la clave de agregación y de join)
df_n = df_n.filter(F.col("COD_MPIO").isNotNull() & (F.col("COD_MPIO") != ""))
n1 = df_n.count()
print(f"  Tras F1 (COD_MPIO no nulo)             : {n1:>10,}  (eliminó {n0-n1:,})")

# F2 — Grupo en valores válidos del SISBEN
df_n = df_n.filter(F.col("Grupo").isin("A","B","C","D"))
n2 = df_n.count()
print(f"  Tras F2 (Grupo IN A/B/C/D)             : {n2:>10,}  (eliminó {n1-n2:,})")
print(f"Total eliminado por filtros: {n0-n2:,} ({100*(n0-n2)/n0:.2f}%)")

Personas antes de filtros: 4,465,955


  Tras F1 (COD_MPIO no nulo)             :  4,465,955  (eliminó 0)


  Tras F2 (Grupo IN A/B/C/D)             :  4,465,955  (eliminó 0)
Total eliminado por filtros: 0 (0.00%)


## 4. Agregación por municipio (T4 + T5)

In [4]:
agg_exprs = [
    F.count("*").alias("n_personas"),
    F.countDistinct(F.concat_ws("|", F.col("CORTE"), F.col("COD_MPIO"), F.col("HOGAR"))).alias("n_hogares"),
    F.sum("FEX").alias("fex_total"),
    F.avg(F.when(F.col("ZONA")==2, 1.0).otherwise(0.0)).alias("pct_rural"),
    F.avg(F.when(F.col("Grupo")=="A", 1.0).otherwise(0.0)).alias("pct_grupo_A"),
    F.avg(F.when(F.col("Grupo")=="B", 1.0).otherwise(0.0)).alias("pct_grupo_B"),
    F.avg(F.when(F.col("Grupo")=="C", 1.0).otherwise(0.0)).alias("pct_grupo_C"),
    F.avg(F.when(F.col("Grupo")=="D", 1.0).otherwise(0.0)).alias("pct_grupo_D"),
]
for i in indicadores:
    agg_exprs.append(F.avg(F.col(i)).alias(f"avg_{i}"))

sisben_mpio = (
    df_n.groupBy("COD_MPIO")
    .agg(*agg_exprs)
)

# Índice de privación: suma promedio de los 15 indicadores → 0..15
priv_cols = [F.col(f"avg_I{i}") for i in range(1, 16)]
sisben_mpio = sisben_mpio.withColumn("idx_privacion", sum(priv_cols))

print(f"Municipios distintos: {sisben_mpio.count():,}")
sisben_mpio.select("COD_MPIO","n_personas","n_hogares","pct_rural","pct_grupo_A","idx_privacion").show(10)

Municipios distintos: 1,099


26/05/22 08:16:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+----------+---------+------------------+-------------------+------------------+
|COD_MPIO|n_personas|n_hogares|         pct_rural|        pct_grupo_A|     idx_privacion|
+--------+----------+---------+------------------+-------------------+------------------+
|   47960|      5036|        4| 0.506155679110405| 0.5645353455123113| 5.453137410643366|
|   23815|      9714|        6|0.6404158945851348| 0.6254889849701462|5.7293596870496195|
|   05380|      2435|        2|0.5071868583162218|0.05790554414784394|1.9552361396303901|
|   19137|      3466|        5|0.8335256780150029|  0.533467974610502| 3.178880553952683|
|   91001|      4793|        6|0.2207385770915919| 0.3559357396202796| 4.810139787189652|
|   05658|      1984|        5|0.3487903225806452| 0.2933467741935484|          3.984375|
|   25823|      1888|        3|0.8140889830508474| 0.3146186440677966| 4.279131355932204|
|   50606|      3980|        4|0.4439698492462312|0.17763819095477387| 2.904020100502513|
|   44560|

## 5. Estadísticos del índice de privación

In [5]:
sisben_mpio.select("idx_privacion","pct_grupo_A","pct_rural").describe().show()

+-------+------------------+--------------------+--------------------+
|summary|     idx_privacion|         pct_grupo_A|           pct_rural|
+-------+------------------+--------------------+--------------------+
|  count|              1099|                1099|                1099|
|   mean|3.8698503626763165|  0.3855728299045176|  0.5643779454618066|
| stddev|0.8933616207827746| 0.17144217440630036| 0.13782430571194845|
|    min|1.4882736156351792|0.024186046511627906|4.390779363336992E-4|
|    max| 7.211160714285715|  0.8410262860017608|  0.9555214723926381|
+-------+------------------+--------------------+--------------------+



## 6. Escribir Silver Parquet

In [6]:
import time
t0 = time.time()
(sisben_mpio.write
    .mode("overwrite")
    .option("compression","snappy")
    .parquet(P.SILVER_SISBEN_MPIO))
print(f"Escrito en {time.time()-t0:.1f}s")

sv = spark.read.parquet(P.SILVER_SISBEN_MPIO)
print(f"Verificación: {sv.count():,} filas (1 por municipio)")
sv.orderBy(F.desc("idx_privacion")).show(10)

Escrito en 15.0s


Verificación: 1,099 filas (1 por municipio)


+--------+----------+---------+------------------+------------------+------------------+-------------------+--------------------+--------------------+------------------+-------------------+-------------------+-------------------+-------------------+--------------------+-------------------+------------------+-------------------+--------------------+------------------+------------------+--------------------+--------------------+-------------------+------------------+
|COD_MPIO|n_personas|n_hogares|         fex_total|         pct_rural|       pct_grupo_A|        pct_grupo_B|         pct_grupo_C|         pct_grupo_D|            avg_I1|             avg_I2|             avg_I3|             avg_I4|             avg_I5|              avg_I6|             avg_I7|            avg_I8|             avg_I9|             avg_I10|           avg_I11|           avg_I12|             avg_I13|             avg_I14|            avg_I15|     idx_privacion|
+--------+----------+---------+------------------+----------

In [7]:
spark.stop()